# Exploratory Data Analysis (EDA)
### Interactive Notebook for AI/ML Interview Preparation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
np.random.seed(42)
print('Libraries loaded!')

---
## 1. Create and Explore a Dataset

In [ ]:
# Synthetic employee dataset
n = 300
df = pd.DataFrame({
    'age': np.random.normal(35, 8, n).clip(22, 65).astype(int),
    'salary': np.random.lognormal(10.8, 0.4, n).astype(int),
    'experience': np.random.poisson(8, n),
    'department': np.random.choice(['Engineering','Sales','Marketing','HR'], n, p=[0.4,0.3,0.2,0.1]),
    'performance': np.random.choice(['Low','Medium','High'], n, p=[0.2,0.5,0.3]),
})
df['salary'] = (df['salary'] * (1 + df['experience']*0.02)).astype(int)

print(df.shape)
print(df.describe())
print(f'\nDtypes:\n{df.dtypes}')

---
## 2. Distribution Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0,0].hist(df['age'], bins=20, edgecolor='black', color='steelblue')
axes[0,0].set_title('Age Distribution')
axes[0,1].hist(df['salary'], bins=30, edgecolor='black', color='coral')
axes[0,1].set_title('Salary Distribution (right-skewed)')
sns.boxplot(data=df, x='department', y='salary', ax=axes[1,0])
axes[1,0].set_title('Salary by Department')
axes[1,0].tick_params(axis='x', rotation=45)
sns.countplot(data=df, x='performance', order=['Low','Medium','High'], ax=axes[1,1])
axes[1,1].set_title('Performance Distribution')
plt.tight_layout(); plt.show()

---
## 3. Correlation Analysis

In [ ]:
# Correlation heatmap
numeric_cols = df.select_dtypes(include=[np.number])
corr = numeric_cols.corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=ax)
ax.set_title('Correlation Matrix')
plt.tight_layout(); plt.show()

---
## 4. Skewness, Kurtosis, and VIF

In [ ]:
from scipy import stats as sp_stats

for col in ['age', 'salary', 'experience']:
    skew = sp_stats.skew(df[col])
    kurt = sp_stats.kurtosis(df[col])
    print(f'{col:12s} — Skewness: {skew:+.2f}, Kurtosis: {kurt:+.2f}')

# VIF for multicollinearity
from numpy.linalg import inv
X = numeric_cols.values
X_std = (X - X.mean(axis=0)) / X.std(axis=0)
corr_matrix = np.corrcoef(X_std.T)
try:
    vif = np.diag(inv(corr_matrix))
    print(f'\nVIF values:')
    for col, v in zip(numeric_cols.columns, vif):
        print(f'  {col}: {v:.2f} {"⚠ HIGH" if v > 5 else "✓ OK"}')
except:
    print('Could not compute VIF (singular matrix)')

---
## Key Interview Takeaways

1. **Always start with shape, dtypes, describe()** before modeling
2. **Visualize distributions** — histograms for continuous, bar plots for categorical
3. **Check correlations** — heatmap reveals relationships and multicollinearity
4. **Skewness > 1** suggests log transform; kurtosis > 3 means heavy tails
5. **VIF > 5** indicates problematic multicollinearity